
# Conventional-chart SG lattice mask stress test

This notebook stress-tests the new KLDMPlus DiffCSP++ conventional-chart lattice symmetry loss on a 10% validation subset.

It checks:

- validation data fields: `l`, `space_group`, `conv_C`, `conv_weight`, `conv_fit_error`
- family-balanced subset coverage
- GT primitive direct residual vs conventional-chart residual
- random/noisy `k` perturbation losses
- optional model-predicted clean `k0` loss using `/workspace/artifacts/HPC/checkpoints/epoch_8900.pt`

The intended success pattern is:

- `conv_weight_mean` close to `1.0`
- `conv_gt_residual` much lower than primitive direct GT residual
- `conv_gt_loss` near zero for most families
- noisy/random `k` has larger conventional loss than GT
- loaded model predictions produce finite `loss_conv_sg` and logs


In [1]:

from __future__ import annotations

import os
import sys
from pathlib import Path
from collections import defaultdict
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    # Useful when opened from notebooks/ directly.
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from torch.utils.data import DataLoader, Subset

from kldmPlus.data import CSPTask, resolve_data_root
from kldmPlus.symmetry.latticeSymmetry import LatticeSymmetry
from kldmPlus.utils.model_loader import build_model, load_checkpoint
from kldmPlus.utils.time import sample_times

try:
    from IPython.display import display
except Exception:
    display = print

CONFIG_PATH = REPO_ROOT / 'configs/kldm_plus/mp_20/mp20_plus_conv_sg_k_x0_em_quad_power_ema_a100.yaml'
CHECKPOINT_PATH = Path('/workspace/artifacts/HPC/checkpoints/epoch_8900.pt')
SPLIT = 'val'
SUBSET_FRACTION = 0.10
SUBSET_SEED = 123
BATCH_SIZE = 64
MAX_MODEL_BATCHES = 4
NOISE_LEVELS = [0.01, 0.05, 0.10, 0.25, 0.50]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('repo=', REPO_ROOT)
print('device=', DEVICE)
print('config=', CONFIG_PATH)
print('checkpoint_exists=', CHECKPOINT_PATH.exists(), CHECKPOINT_PATH)


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
repo= /workspace
device= cpu
config= /workspace/configs/kldm_plus/mp_20/mp20_plus_conv_sg_k_x0_em_quad_power_ema_a100.yaml
checkpoint_exists= True /workspace/artifacts/HPC/checkpoints/epoch_8900.pt


In [2]:

def family_from_sg(sg: int) -> str:
    sg = int(sg)
    if 1 <= sg <= 2:
        return 'triclinic'
    if 3 <= sg <= 15:
        return 'monoclinic'
    if 16 <= sg <= 74:
        return 'orthorhombic'
    if 75 <= sg <= 142:
        return 'tetragonal'
    if 143 <= sg <= 194:
        return 'hexagonal_trigonal'
    if 195 <= sg <= 230:
        return 'cubic'
    return 'unknown'


def make_balanced_subset(dataset, *, subset_size: int, seed: int, group_key):
    grouped = defaultdict(list)
    for idx in range(len(dataset)):
        grouped[group_key(dataset[idx])].append(idx)
    rng = np.random.default_rng(seed)
    groups = sorted(grouped)
    shuffled = []
    for group in groups:
        arr = np.array(grouped[group], dtype=np.int64)
        rng.shuffle(arr)
        shuffled.append((group, arr.tolist()))

    selected = []
    round_idx = 0
    while len(selected) < subset_size:
        progressed = False
        for _, indices in shuffled:
            if round_idx < len(indices):
                selected.append(indices[round_idx])
                progressed = True
                if len(selected) >= subset_size:
                    break
        if not progressed:
            break
        round_idx += 1
    return Subset(dataset, selected)


def tensor_to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def weighted_mean(values: torch.Tensor, weights: torch.Tensor) -> torch.Tensor:
    denom = weights.sum().clamp_min(1.0)
    return (values * weights).sum() / denom


def add_family_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'space_group' in df.columns:
        df['family'] = df['space_group'].map(family_from_sg)
    return df


def summarize_by_family(df: pd.DataFrame, value_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    if 'family' not in df.columns:
        raise KeyError("summarize_by_family requires a 'family' column.")
    if '_row_count' not in df.columns:
        df['_row_count'] = 1
    present = [col for col in value_cols if col in df.columns]
    aggregations = {'n': ('_row_count', 'sum')}
    for col in present:
        aggregations[f'{col}_mean'] = (col, 'mean')
        aggregations[f'{col}_p95'] = (col, lambda s: float(np.nanquantile(s, 0.95)))
    return (
        df.groupby('family', dropna=False)
        .agg(**aggregations)
        .reset_index()
        .sort_values('family')
    )


def conventional_projection_volume_metrics(
    sym: LatticeSymmetry,
    primitive_k: torch.Tensor,
    conv_C: torch.Tensor,
    space_group: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """Return conventional volume before/after SG projection for primitive k."""
    primitive_k = primitive_k.reshape(-1, 6)
    conv_C = conv_C.reshape(-1, 3, 3).to(device=primitive_k.device, dtype=primitive_k.dtype)
    space_group = space_group.reshape(-1).to(device=primitive_k.device)
    primitive_cell = sym.v2m(primitive_k)
    conventional_cell = conv_C @ primitive_cell
    conventional_k = sym.m2v(sym.de_so3(conventional_cell))
    projected_k = sym.proj_k_to_spacegroup(conventional_k, space_group)
    projected_cell = sym.v2m(projected_k)
    conventional_volume = torch.linalg.det(conventional_cell).abs()
    projected_volume = torch.linalg.det(projected_cell).abs()
    volume_ratio = projected_volume / conventional_volume.clamp_min(1.0e-12)
    return {
        'conventional_volume': conventional_volume,
        'projected_volume': projected_volume,
        'volume_ratio': volume_ratio,
        'volume_abs_rel_error': (volume_ratio - 1.0).abs(),
    }


In [3]:

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

assert config['model']['lattice_representation'] == 'diffcsp_k'
assert config['model']['lattice_parameterization'] == 'x0'
assert float(config['model']['lambda_l']) == 1.0
assert float(config['model']['lattice_sg_lambda']) == 0.0
assert bool(config['model']['conv_sg_aux']['enabled']) is True
assert float(config['model']['conv_sg_aux']['lambda']) == 1.0

print('experiment=', config['experiment_name'])
print('lambda_l=', config['model']['lambda_l'])
print('lambda_conv_sg=', config['model']['conv_sg_aux']['lambda'])
print('primitive_lattice_sg_lambda=', config['model']['lattice_sg_lambda'])


experiment= plus_mp20_conv_sg_k_x0_em_quad_power_ema_a100
lambda_l= 1.0
lambda_conv_sg= 1.0
primitive_lattice_sg_lambda= 0.0


In [4]:

# Load the validation split through the same KLDMPlus diffcsp_k transform used by training.
dataset_cfg = config['dataset']
model_cfg = config['model']
root = resolve_data_root(dataset_cfg.get('root'))

task = CSPTask(
    dataset_name=str(dataset_cfg['name']),
    lattice_parameterization=str(model_cfg['lattice_parameterization']),
    lattice_representation=str(dataset_cfg['lattice_representation']),
)

dataset_full = task.fit_dataset(root=root, split=SPLIT, download=True)
subset_size = max(1, int(round(len(dataset_full) * SUBSET_FRACTION)))
dataset = make_balanced_subset(
    dataset_full,
    subset_size=subset_size,
    seed=SUBSET_SEED,
    group_key=lambda sample: family_from_sg(int(torch.as_tensor(sample.space_group).reshape(-1)[0].item())),
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=dataset_full.collate_fn,
)

print('full_val_size=', len(dataset_full))
print('subset_size=', len(dataset))
print('num_batches=', len(loader))


dataset_cache action=load dataset=mp_20 split=val reason=fresh path=/workspace/data/mp_20/processed/val
dataset_cache action=from_cache_path:start dataset=mp_20 split=val
dataset_cache action=from_cache_path:done dataset=mp_20 split=val
dataset_cache action=builder_build:start dataset=mp_20 split=val
dataset_cache action=builder_build:done dataset=mp_20 split=val


/workspace/src/kldmPlus/data/transform.py:621: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  conv_cell = torch.as_tensor(
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact

full_val_size= 9047
subset_size= 905
num_batches= 15


In [5]:

# Basic data-field and family coverage diagnostics.
rows = []
for local_idx, sample in enumerate(dataset):
    sg = int(torch.as_tensor(sample.space_group).reshape(-1)[0].item())
    rows.append({
        'graph_index': local_idx,
        'space_group': sg,
        'family': family_from_sg(sg),
        'has_l': hasattr(sample, 'l'),
        'has_conv_C': hasattr(sample, 'conv_C'),
        'has_conv_weight': hasattr(sample, 'conv_weight'),
        'conv_weight': float(torch.as_tensor(getattr(sample, 'conv_weight', torch.tensor([0.0]))).reshape(-1)[0].item()),
        'conv_fit_error': float(torch.as_tensor(getattr(sample, 'conv_fit_error', torch.tensor([float('nan')]))).reshape(-1)[0].item()),
        'num_atoms': int(len(sample.pos)),
    })

df_fields = pd.DataFrame(rows)
display(df_fields.groupby('family').agg(
    n=('graph_index', 'count'),
    conv_weight_mean=('conv_weight', 'mean'),
    conv_fit_error_median=('conv_fit_error', 'median'),
    conv_fit_error_p95=('conv_fit_error', lambda s: float(np.nanquantile(s, 0.95))),
    atoms_mean=('num_atoms', 'mean'),
).reset_index().sort_values('family'))

display(df_fields.head())

assert df_fields['has_l'].all(), 'Missing sample.l from diffcsp_k transform.'
assert df_fields['has_conv_C'].all(), 'Missing sample.conv_C. Rebuild processed cache or verify transform.'
assert df_fields['has_conv_weight'].all(), 'Missing sample.conv_weight. Rebuild processed cache or verify transform.'


,family,n,conv_weight_mean,conv_fit_error_median,conv_fit_error_p95,atoms_mean
0,cubic,151,1.0,1.430511e-06,0.000003,7.284768
1,hexagonal_trigonal,151,1.0,1.058280e-06,0.000008,9.569536
2,monoclinic,151,1.0,1.907349e-06,0.000005,13.913907
3,orthorhombic,151,1.0,9.536743e-07,0.000003,12.245033
4,tetragonal,151,1.0,1.102125e-06,0.000004,9.172185
5,triclinic,150,1.0,1.430511e-06,0.000003,16.853333


,graph_index,space_group,family,has_l,has_conv_C,has_conv_weight,conv_weight,conv_fit_error,num_atoms
0,0,225,cubic,True,True,True,1.0,2.441579e-06,10
1,1,189,hexagonal_trigonal,True,True,True,1.0,6.654781e-07,9
2,2,12,monoclinic,True,True,True,1.0,1.928723e-06,11
3,3,62,orthorhombic,True,True,True,1.0,9.516832e-07,12
4,4,123,tetragonal,True,True,True,1.0,4.810407e-07,16


In [6]:

# Operator-level stress tests: GT k, noisy GT k, pure random k.
sym = LatticeSymmetry().to(DEVICE)
rows = []
noise_rng = torch.Generator(device=DEVICE).manual_seed(2027)

graph_offset = 0
with torch.no_grad():
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(DEVICE)
        sg = batch.space_group.reshape(-1).long()
        conv_C = batch.conv_C.reshape(-1, 3, 3)
        conv_weight = batch.conv_weight.reshape(-1).to(dtype=batch.l.dtype)
        gt_k = batch.l.reshape(-1, 6)
        family = [family_from_sg(int(v)) for v in sg.detach().cpu().tolist()]

        primitive_gt_res = sym.direct_sg_residual_abs_mean(gt_k, sg)
        conv_gt_res = sym.conventional_sg_residual_abs_mean(gt_k, conv_C, sg)
        conv_gt_loss = sym.conventional_sg_loss_per_graph(gt_k, conv_C, sg)

        variants = [('gt', gt_k)]
        for sigma in NOISE_LEVELS:
            variants.append((f'gt_plus_noise_{sigma:g}', gt_k + sigma * torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=noise_rng)))
        variants.append(('random_normal_unit', torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=noise_rng)))
        variants.append(('random_like_gt_scale', gt_k.mean(dim=0, keepdim=True) + gt_k.std(dim=0, keepdim=True).clamp_min(1e-6) * torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=noise_rng)))

        for variant_name, k_variant in variants:
            conv_res = sym.conventional_sg_residual_abs_mean(k_variant, conv_C, sg)
            conv_loss = sym.conventional_sg_loss_per_graph(k_variant, conv_C, sg)
            primitive_res = sym.direct_sg_residual_abs_mean(k_variant, sg)
            for i in range(gt_k.shape[0]):
                rows.append({
                    'graph_index': graph_offset + i,
                    'batch': batch_idx,
                    'variant': variant_name,
                    'space_group': int(sg[i].item()),
                    'family': family[i],
                    'conv_weight': float(conv_weight[i].item()),
                    'primitive_residual': float(primitive_res[i].item()),
                    'conv_residual': float(conv_res[i].item()),
                    'conv_loss': float(conv_loss[i].item()),
                    'primitive_gt_residual': float(primitive_gt_res[i].item()),
                    'conv_gt_residual': float(conv_gt_res[i].item()),
                    'conv_gt_loss': float(conv_gt_loss[i].item()),
                })
        graph_offset += gt_k.shape[0]

df_stress = pd.DataFrame(rows)
display(df_stress.groupby('variant').agg(
    n=('graph_index', 'count'),
    conv_weight_mean=('conv_weight', 'mean'),
    primitive_residual_mean=('primitive_residual', 'mean'),
    conv_residual_mean=('conv_residual', 'mean'),
    conv_residual_p95=('conv_residual', lambda s: float(np.nanquantile(s, 0.95))),
    conv_loss_mean=('conv_loss', 'mean'),
    conv_loss_p95=('conv_loss', lambda s: float(np.nanquantile(s, 0.95))),
).reset_index().sort_values('variant'))

gt_df = df_stress[df_stress['variant'] == 'gt'].copy()
display(summarize_by_family(gt_df, ['primitive_residual', 'conv_residual', 'conv_loss', 'conv_weight']))


,variant,n,conv_weight_mean,primitive_residual_mean,conv_residual_mean,conv_residual_p95,conv_loss_mean,conv_loss_p95
0,gt,905,1.0,0.069182,8.754227e-08,2.968857e-07,4.461764e-14,2.151516e-13
1,gt_plus_noise_0.01,905,1.0,0.071342,4.004963e-03,8.906095e-03,5.017531e-05,1.502406e-04
2,gt_plus_noise_0.05,905,1.0,0.079308,1.951579e-02,4.334249e-02,1.191068e-03,3.624596e-03
3,gt_plus_noise_0.1,905,1.0,0.090992,4.045699e-02,8.798984e-02,5.141219e-03,1.444077e-02
4,gt_plus_noise_0.25,905,1.0,0.130990,9.677238e-02,2.124310e-01,2.955142e-02,8.730603e-02
5,gt_plus_noise_0.5,905,1.0,0.222209,1.976528e-01,4.370255e-01,1.235958e-01,3.680171e-01
6,random_like_gt_scale,905,1.0,0.094714,1.152024e-01,2.797058e-01,4.243378e-02,1.331549e-01
7,random_normal_unit,905,1.0,0.404279,4.029491e-01,8.951860e-01,5.100518e-01,1.488266e+00


,family,n,primitive_residual_mean,primitive_residual_p95,conv_residual_mean,conv_residual_p95,conv_loss_mean,conv_loss_p95,conv_weight_mean,conv_weight_p95
0,cubic,151,0.135485,0.163377,1.912536e-07,3.955826e-07,1.014512e-13,3.523194e-13,1.0,1.0
1,hexagonal_trigonal,151,0.126224,0.227019,8.537453e-08,2.054081e-07,2.642829e-14,1.047454e-13,1.0,1.0
2,monoclinic,151,0.044055,0.097743,6.910220e-08,1.793361e-07,3.781234e-14,1.499884e-13,1.0,1.0
3,orthorhombic,151,0.038621,0.163640,3.292512e-08,1.565520e-07,1.529253e-14,6.879836e-14,1.0,1.0
4,tetragonal,151,0.070252,0.161090,1.460184e-07,3.440189e-07,8.642600e-14,2.908740e-13,1.0,1.0
5,triclinic,150,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,1.0


In [7]:
# Volume-safety diagnostics for the conventional-chart projection.
# The projection should preserve the isotropic/log-volume degree of freedom.
volume_rows = []
volume_rng = torch.Generator(device=DEVICE).manual_seed(2031)

graph_offset = 0
with torch.no_grad():
    for batch_idx, batch in enumerate(loader):
        batch = batch.to(DEVICE)
        sg = batch.space_group.reshape(-1).long()
        conv_C = batch.conv_C.reshape(-1, 3, 3)
        conv_weight = batch.conv_weight.reshape(-1).to(dtype=batch.l.dtype)
        gt_k = batch.l.reshape(-1, 6)
        variants = [
            ('gt', gt_k),
            ('gt_plus_noise_0.05', gt_k + 0.05 * torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=volume_rng)),
            ('gt_plus_noise_0.10', gt_k + 0.10 * torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=volume_rng)),
            ('random_normal_unit', torch.randn(gt_k.shape, device=DEVICE, dtype=gt_k.dtype, generator=volume_rng)),
        ]
        for variant_name, k_variant in variants:
            metrics = conventional_projection_volume_metrics(sym, k_variant, conv_C, sg)
            for i, sg_i in enumerate(sg.detach().cpu().tolist()):
                volume_rows.append({
                    'graph_index': graph_offset + i,
                    'batch': batch_idx,
                    'variant': variant_name,
                    'space_group': int(sg_i),
                    'family': family_from_sg(int(sg_i)),
                    'conv_weight': float(conv_weight[i].detach().cpu().item()),
                    'conventional_volume': float(metrics['conventional_volume'][i].detach().cpu().item()),
                    'projected_volume': float(metrics['projected_volume'][i].detach().cpu().item()),
                    'volume_ratio': float(metrics['volume_ratio'][i].detach().cpu().item()),
                    'volume_abs_rel_error': float(metrics['volume_abs_rel_error'][i].detach().cpu().item()),
                })
        graph_offset += gt_k.shape[0]

df_volume = pd.DataFrame(volume_rows)
display(df_volume.groupby('variant').agg(
    n=('graph_index', 'count'),
    conv_weight_mean=('conv_weight', 'mean'),
    volume_ratio_mean=('volume_ratio', 'mean'),
    volume_ratio_p05=('volume_ratio', lambda s: float(np.nanquantile(s, 0.05))),
    volume_ratio_p95=('volume_ratio', lambda s: float(np.nanquantile(s, 0.95))),
    volume_abs_rel_error_mean=('volume_abs_rel_error', 'mean'),
    volume_abs_rel_error_p95=('volume_abs_rel_error', lambda s: float(np.nanquantile(s, 0.95))),
    volume_abs_rel_error_max=('volume_abs_rel_error', 'max'),
).reset_index().sort_values('variant'))

gt_volume = df_volume[df_volume['variant'] == 'gt'].copy()
display(summarize_by_family(gt_volume, ['volume_ratio', 'volume_abs_rel_error', 'conv_weight']))

print('PASS_gt_projection_volume_safe_mean=', float(gt_volume['volume_abs_rel_error'].mean()) < 1e-5)
print('PASS_gt_projection_volume_safe_p95=', float(np.nanquantile(gt_volume['volume_abs_rel_error'], 0.95)) < 1e-5)


,variant,n,conv_weight_mean,volume_ratio_mean,volume_ratio_p05,volume_ratio_p95,volume_abs_rel_error_mean,volume_abs_rel_error_p95,volume_abs_rel_error_max
0,gt,905,1.0,1.0000,0.999998,1.000001,6.319410e-07,0.000002,0.000003
1,gt_plus_noise_0.05,905,1.0,1.0000,0.999998,1.000002,9.675052e-07,0.000002,0.000020
2,gt_plus_noise_0.10,905,1.0,1.0000,0.999998,1.000002,9.082958e-07,0.000002,0.000014
3,random_normal_unit,905,1.0,1.0001,0.999926,1.000070,1.674485e-04,0.000184,0.070975


,family,n,volume_ratio_mean,volume_ratio_p95,volume_abs_rel_error_mean,volume_abs_rel_error_p95,conv_weight_mean,conv_weight_p95
0,cubic,151,1.0,1.000001,6.584142e-07,0.000002,1.0,1.0
1,hexagonal_trigonal,151,1.0,1.000000,4.646004e-07,0.000001,1.0,1.0
2,monoclinic,151,1.0,1.000001,6.970980e-07,0.000002,1.0,1.0
3,orthorhombic,151,1.0,1.000001,4.685478e-07,0.000001,1.0,1.0
4,tetragonal,151,1.0,1.000001,6.738088e-07,0.000002,1.0,1.0
5,triclinic,150,1.0,1.000001,8.304914e-07,0.000002,1.0,1.0


PASS_gt_projection_volume_safe_mean= True
PASS_gt_projection_volume_safe_p95= True


In [8]:

# Pass/fail-style interpretation for the data/operator part.
gt_valid = gt_df[gt_df['conv_weight'] > 0.5].copy()
if len(gt_valid) == 0:
    print('FAIL: no valid conventional transforms. Check pymatgen availability and cache rebuild.')
else:
    primitive_mean = float(gt_valid['primitive_residual'].mean())
    conv_mean = float(gt_valid['conv_residual'].mean())
    conv_p95 = float(np.nanquantile(gt_valid['conv_residual'], 0.95))
    weight_mean = float(gt_df['conv_weight'].mean())
    print('valid_graphs=', len(gt_valid), '/', len(gt_df))
    print('conv_weight_mean=', weight_mean)
    print('primitive_gt_residual_mean=', primitive_mean)
    print('conv_gt_residual_mean=', conv_mean)
    print('conv_gt_residual_p95=', conv_p95)
    print('ratio_conv_over_primitive=', conv_mean / max(primitive_mean, 1e-12))
    print('PASS_conv_gt_lower_than_primitive=', conv_mean < primitive_mean)
    print('PASS_conv_weight_reasonable=', weight_mean > 0.8)
    print('PASS_conv_gt_smallish=', conv_mean < 0.02)


valid_graphs= 905 / 905
conv_weight_mean= 1.0
primitive_gt_residual_mean= 0.06918248187587052
conv_gt_residual_mean= 8.754226965254884e-08
conv_gt_residual_p95= 2.9688574159081317e-07
ratio_conv_over_primitive= 1.2653820342788518e-06
PASS_conv_gt_lower_than_primitive= True
PASS_conv_weight_reasonable= True
PASS_conv_gt_smallish= True


In [9]:

# Optional model-prediction stress test with /workspace/artifacts/HPC/checkpoints/epoch_8900.pt.
# This uses the same Algorithm-2 loss path that training uses.
model = None
checkpoint_status = 'not_attempted'
try:
    model = build_model(config=config, device=DEVICE)
    if CHECKPOINT_PATH.exists():
        load_checkpoint(
            checkpoint_path=CHECKPOINT_PATH,
            model=model,
            device=DEVICE,
            prefer_ema_weights=True,
        )
        checkpoint_status = 'loaded'
    else:
        checkpoint_status = 'missing_checkpoint_using_random_init'
    model.eval()
except Exception as exc:
    checkpoint_status = f'failed: {type(exc).__name__}: {exc}'
    model = None

print('checkpoint_status=', checkpoint_status)
print('model_available=', model is not None)


checkpoint_status= loaded
model_available= True


In [ ]:

model_rows = []
if model is None:
    print('Skipping model stress test because model/checkpoint could not be constructed.')
else:
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= MAX_MODEL_BATCHES:
                break
            batch = batch.to(DEVICE)
            t = sample_times(batch, lower_bound=1e-3)
            loss, metrics = model.algorithm2_loss(batch=batch, t=t, debug=False)
            model_rows.append({
                'batch': batch_idx,
                'num_graphs': int(batch.num_graphs),
                'loss': float(loss.detach().cpu().item()),
                'loss_l': float(metrics['loss_l'].detach().cpu().item()),
                'loss_conv_sg': float(metrics['loss_conv_sg'].detach().cpu().item()),
                'loss_conv_sg_weighted': float(metrics['loss_conv_sg_weighted'].detach().cpu().item()),
                'loss_conv_sg_lambda_scaled': float(metrics['loss_conv_sg_lambda_scaled'].detach().cpu().item()),
                'lambda_conv_sg': float(metrics['lambda_conv_sg'].detach().cpu().item()),
                'conv_weight_mean': float(metrics['conv_weight_mean'].detach().cpu().item()),
                'primitive_projection_error_pred_k': float(metrics['primitive_projection_error_pred_k'].detach().cpu().item()),
                'primitive_projection_error_gt_k': float(metrics['primitive_projection_error_gt_k'].detach().cpu().item()),
                'conv_projection_error_pred_k': float(metrics['conv_projection_error_pred_k'].detach().cpu().item()),
                'conv_projection_error_gt_k': float(metrics['conv_projection_error_gt_k'].detach().cpu().item()),
                'conv_sg_time_weight_mean': float(metrics['conv_sg_time_weight_mean'].detach().cpu().item()),
            })

df_model = pd.DataFrame(model_rows)
if len(df_model):
    display(df_model)
    display(df_model.describe(include='all'))
    assert np.isfinite(df_model['loss']).all()
    assert np.isfinite(df_model['loss_conv_sg']).all()
    assert (df_model['lambda_conv_sg'] == 1.0).all()
    assert (df_model['conv_weight_mean'] > 0.0).all()
else:
    print('No model rows produced.')


In [ ]:

# Optional: collect per-family model predicted clean-k residuals by directly running the noisy forward target + score network.
# This is useful if you want family-specific model residuals, not just batch averages.
model_family_rows = []
if model is None:
    print('Skipping per-family predicted-k diagnostics.')
else:
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= MAX_MODEL_BATCHES:
                break
            batch = batch.to(DEVICE)
            t = sample_times(batch, lower_bound=1e-3)
            times = __import__('kldmPlus.utils.time', fromlist=['make_times']).make_times(batch, t)
            (v_t, f_t, l_t), (_, target_l) = model.algorithm1_training_targets(batch=batch, times=times)
            preds = model.score_network(
                t=times.graph,
                pos=f_t,
                v=v_t,
                h=batch.atomic_numbers,
                l=l_t,
                node_index=batch.batch,
                edge_node_index=batch.edge_node_index,
            )
            pred_l = preds['l']
            sg = batch.space_group.reshape(-1).long()
            conv_C = batch.conv_C.reshape(-1, 3, 3)
            conv_weight = batch.conv_weight.reshape(-1).to(dtype=pred_l.dtype)
            pred_conv_res = model.lattice_symmetry.conventional_sg_residual_abs_mean(pred_l, conv_C, sg)
            gt_conv_res = model.lattice_symmetry.conventional_sg_residual_abs_mean(target_l, conv_C, sg)
            pred_conv_loss = model.lattice_symmetry.conventional_sg_loss_per_graph(pred_l, conv_C, sg)
            for i, sg_i in enumerate(sg.detach().cpu().tolist()):
                model_family_rows.append({
                    'graph_index': len(model_family_rows),
                    'batch': batch_idx,
                    'space_group': int(sg_i),
                    'family': family_from_sg(int(sg_i)),
                    'conv_weight': float(conv_weight[i].detach().cpu().item()),
                    'pred_conv_residual': float(pred_conv_res[i].detach().cpu().item()),
                    'gt_conv_residual': float(gt_conv_res[i].detach().cpu().item()),
                    'pred_conv_loss': float(pred_conv_loss[i].detach().cpu().item()),
                })

df_model_family = pd.DataFrame(model_family_rows)
if len(df_model_family):
    display(summarize_by_family(df_model_family, ['pred_conv_residual', 'gt_conv_residual', 'pred_conv_loss', 'conv_weight']))
else:
    print('No per-family model rows produced.')


,family,n,pred_conv_residual_mean,pred_conv_residual_p95,gt_conv_residual_mean,gt_conv_residual_p95,pred_conv_loss_mean,pred_conv_loss_p95,conv_weight_mean,conv_weight_p95
0,cubic,43,1.213603,2.192947,2.003373e-07,3.902760e-07,4.180624,7.904282,1.0,1.0
1,hexagonal_trigonal,43,0.864925,1.827399,1.121739e-07,2.395107e-07,2.405139,7.802443,1.0,1.0
2,monoclinic,43,0.530915,1.844170,6.160031e-08,1.594898e-07,2.418554,17.170189,1.0,1.0
3,orthorhombic,43,1.021865,3.246063,2.150875e-08,7.127505e-08,4.920437,22.358284,1.0,1.0
4,tetragonal,42,0.867164,2.759441,1.355100e-07,3.340024e-07,3.410629,15.720100,1.0,1.0
5,triclinic,42,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,1.0,1.0


In [ ]:
# Model-predicted volume-safety diagnostics.
model_volume_rows = []
if model is None:
    print('Skipping model volume diagnostics.')
else:
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            if batch_idx >= MAX_MODEL_BATCHES:
                break
            batch = batch.to(DEVICE)
            t = sample_times(batch, lower_bound=1e-3)
            times = __import__('kldmPlus.utils.time', fromlist=['make_times']).make_times(batch, t)
            (v_t, f_t, l_t), (_, target_l) = model.algorithm1_training_targets(batch=batch, times=times)
            preds = model.score_network(
                t=times.graph,
                pos=f_t,
                v=v_t,
                h=batch.atomic_numbers,
                l=l_t,
                node_index=batch.batch,
                edge_node_index=batch.edge_node_index,
            )
            pred_l = preds['l']
            sg = batch.space_group.reshape(-1).long()
            conv_C = batch.conv_C.reshape(-1, 3, 3)
            conv_weight = batch.conv_weight.reshape(-1).to(dtype=pred_l.dtype)
            pred_metrics = conventional_projection_volume_metrics(model.lattice_symmetry, pred_l, conv_C, sg)
            gt_metrics = conventional_projection_volume_metrics(model.lattice_symmetry, target_l, conv_C, sg)
            for i, sg_i in enumerate(sg.detach().cpu().tolist()):
                model_volume_rows.append({
                    'graph_index': len(model_volume_rows),
                    'batch': batch_idx,
                    'space_group': int(sg_i),
                    'family': family_from_sg(int(sg_i)),
                    'conv_weight': float(conv_weight[i].detach().cpu().item()),
                    'pred_volume_ratio': float(pred_metrics['volume_ratio'][i].detach().cpu().item()),
                    'pred_volume_abs_rel_error': float(pred_metrics['volume_abs_rel_error'][i].detach().cpu().item()),
                    'gt_volume_ratio': float(gt_metrics['volume_ratio'][i].detach().cpu().item()),
                    'gt_volume_abs_rel_error': float(gt_metrics['volume_abs_rel_error'][i].detach().cpu().item()),
                })

df_model_volume = pd.DataFrame(model_volume_rows)
if len(df_model_volume):
    display(summarize_by_family(df_model_volume, ['pred_volume_ratio', 'pred_volume_abs_rel_error', 'gt_volume_ratio', 'gt_volume_abs_rel_error', 'conv_weight']))
    display(df_model_volume.describe(include='all'))
else:
    print('No model volume rows produced.')



## How to read this notebook

The key table is the `variant` stress table:

- `gt`: should have low `conv_residual` and `conv_loss`, especially compared with `primitive_residual`.
- `gt_plus_noise_*`: should increase gradually as noise increases.
- `random_*`: should be clearly worse than GT.

For model batches:

- `lambda_conv_sg` must be exactly `1.0`.
- `loss_conv_sg_lambda_scaled` should equal `loss_conv_sg_weighted` because lambda is 1.
- `conv_projection_error_gt_k` should be low if the conventional chart transform is doing its job.
- `conv_weight_mean` should be near 1; if it is low, the issue is the data/operator chart, not the neural model.


Volume-safety checks:

- `volume_ratio = |det L_c_proj| / |det L_c|` should be very close to `1.0` for GT.
- `volume_abs_rel_error` should be near numerical precision for GT if the projector is truly shape-only.
- If model-predicted `pred_volume_ratio` also stays close to `1.0`, the auxiliary is not directly asking the model to change volume; it is only constraining conventional-chart shape.
